# Univariate Workflow Solution: Airline Passengers

**Solution notebook** for the exercise in `01_univariate_workflow.ipynb`.

This notebook demonstrates a complete end-to-end univariate time series pipeline
on the classic **airline passengers** dataset (Box-Jenkins, 1970) using **chronobox**.

## Pipeline

1. Exploratory analysis
2. Unit root tests (ADF, KPSS, Zivot-Andrews) on raw and log-transformed series
3. STL decomposition (period=12)
4. Economic filters (HP, Hamilton)
5. Model fitting (6 models including seasonal ARIMA and multiplicative ETS)
6. Out-of-sample comparison (12-month holdout)
7. Model ranking by RMSE
8. Forecasting with the best model
9. Report generation

**Dataset**: `airline.csv` - Monthly airline passengers (1949-1960, 144 obs)

## Pipeline Flowchart

```
airline.csv (monthly, 144 obs)
   |
   v
[1. Exploratory Analysis] ---> Descriptive stats, seasonality plot
   |
   v
[2. Unit Root Tests] -------> ADF, KPSS, Zivot-Andrews (raw + log)
   |                           => Series is I(1) with strong seasonality
   v
[3. STL Decomposition] -----> period=12, robust=True
   |                           => Multiplicative pattern visible
   v
[4. Filters] ---------------> HP filter (lambda=14400 monthly)
   |                           Hamilton filter (h=24, p=12)
   v
[5. Train/Test Split] ------> Last 12 months as test set
   |
   v
[6. Model Fitting] ---------> SARIMA(0,1,1)(0,1,1)[12] (Box-Jenkins classic)
   |                           SARIMA(1,1,0)(1,1,0)[12]
   |                           ETS(M,A,M) multiplicative seasonal
   |                           Auto-ARIMA (seasonal=True, m=12)
   |                           Holt-Winters multiplicative
   |                           Theta method
   v
[7. Model Comparison] ------> AIC, BIC, RMSE, MAE, MAPE ranking
   |                           => Save outputs/univariate_models_comparison.csv
   v
[8. Diagnostics] -----------> Ljung-Box, Jarque-Bera on best model
   |
   v
[9. Forecast] --------------> Best model => 24-month ahead forecast
   |                           => Save outputs/univariate_forecasts.csv
   v
[10. Report] ----------------> generate_comparison_report()
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import warnings
warnings.filterwarnings('ignore')

# chronobox imports
from chronobox import ARIMA, ETS, HoltWinters, ThetaMethod, auto_arima
from chronobox.decomposition import STL
from chronobox.filters import hp_filter, hamilton_filter
from chronobox.tests_stat import (
    adf_test, kpss_test, zivot_andrews_test,
    ljung_box_test, jarque_bera_test
)
from chronobox.visualization import (
    plot_series, plot_decomposition, plot_diagnostics, plot_forecast,
    set_theme
)

# Workflow utilities
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'utils'))
from report_generator import generate_comparison_report

set_theme('professional')

# Paths
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
output_dir = os.path.join(os.path.dirname(os.getcwd()), 'outputs')
os.makedirs(output_dir, exist_ok=True)

print('chronobox imported successfully')
print(f'Data dir: {data_dir}')
print(f'Output dir: {output_dir}')

---
## 1. Data Loading and Exploratory Analysis

The airline dataset is a classic time series with 144 monthly observations of
international airline passengers (in thousands) from January 1949 to December 1960.

Key characteristics:
- **Strong upward trend**: passengers increase over time
- **Multiplicative seasonality**: seasonal amplitude grows with the level
- **Summer peaks**: July-August have highest traffic

In [2]:
# Load airline data
df = pd.read_csv(os.path.join(data_dir, 'airline.csv'), parse_dates=['date'])
df = df.set_index('date')
df.index.freq = 'MS'

y = df['passengers'].values

print(f'Period: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Observations: {len(df)}')
print(f'Frequency: Monthly')
print()
df.head(10)

In [3]:
# Descriptive statistics
print('=== Descriptive Statistics ===')
print(f'Mean:     {np.mean(y):.2f}')
print(f'Std Dev:  {np.std(y):.2f}')
print(f'Min:      {np.min(y):.2f} ({df.index[np.argmin(y)].date()})')
print(f'Max:      {np.max(y):.2f} ({df.index[np.argmax(y)].date()})')
print(f'Skewness: {pd.Series(y).skew():.4f}')
print(f'Kurtosis: {pd.Series(y).kurtosis():.4f}')

# Log-transformed series
y_log = np.log(y)
print(f'\n=== Log-Transformed Statistics ===')
print(f'Mean:     {np.mean(y_log):.4f}')
print(f'Std Dev:  {np.std(y_log):.4f}')

# Monthly averages to see seasonality
monthly_avg = df.groupby(df.index.month)['passengers'].mean()
print(f'\n=== Monthly Averages ===')
for m, v in monthly_avg.items():
    print(f'  Month {m:2d}: {v:.1f}')

In [4]:
# GRAPH 1: Time series + log-transformed + monthly box plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original series
axes[0, 0].plot(df.index, y, color='steelblue', linewidth=1.5)
axes[0, 0].set_title('Airline Passengers (Level)', fontsize=12)
axes[0, 0].set_ylabel('Passengers (thousands)')
axes[0, 0].grid(True, alpha=0.3)

# Log-transformed
axes[0, 1].plot(df.index, y_log, color='darkorange', linewidth=1.5)
axes[0, 1].set_title('Log(Passengers)', fontsize=12)
axes[0, 1].set_ylabel('Log(Passengers)')
axes[0, 1].grid(True, alpha=0.3)

# Monthly boxplot
monthly_data = [df[df.index.month == m]['passengers'].values for m in range(1, 13)]
bp = axes[1, 0].boxplot(monthly_data, labels=[
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.5)
axes[1, 0].set_title('Monthly Distribution', fontsize=12)
axes[1, 0].set_ylabel('Passengers')
axes[1, 0].grid(True, alpha=0.3)

# Year-over-year growth
growth = np.diff(y, 12) / y[:-12] * 100
axes[1, 1].plot(df.index[12:], growth, color='green', linewidth=1)
axes[1, 1].axhline(y=np.mean(growth), color='red', linestyle='--',
                   label=f'Mean: {np.mean(growth):.1f}%')
axes[1, 1].set_title('Year-over-Year Growth (%)', fontsize=12)
axes[1, 1].set_ylabel('Growth (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Airline Passengers - Exploratory Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Unit Root Tests

We test both the raw and log-transformed series. The airline data has:
- A clear upward trend (non-stationary in mean)
- Increasing variance (suggests log transformation)

| Test | H0 | H1 | Interpretation |
|------|----|----|----------------|
| ADF  | Unit root | Stationary | Reject H0 => stationary |
| KPSS | Stationary | Unit root | Reject H0 => non-stationary |
| ZA   | Unit root (no break) | Stationary with break | Reject H0 => stationary with break |

In [5]:
# ADF Test on raw series
adf_raw = adf_test(y, regression='c', autolag='AIC')
print('=== ADF Test: Raw Passengers ===')
print(f'Test Statistic: {adf_raw.statistic:.4f}')
print(f'P-value:        {adf_raw.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Stationary)" if adf_raw.reject_at_5pct else "Fail to reject H0 (Non-stationary)"}')
print(f'Critical Values: {adf_raw.critical_values}')

# ADF Test on log series
adf_log = adf_test(y_log, regression='c', autolag='AIC')
print(f'\n=== ADF Test: Log(Passengers) ===')
print(f'Test Statistic: {adf_log.statistic:.4f}')
print(f'P-value:        {adf_log.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Stationary)" if adf_log.reject_at_5pct else "Fail to reject H0 (Non-stationary)"}')

In [6]:
# KPSS Test
kpss_raw = kpss_test(y, regression='c')
print('=== KPSS Test: Raw Passengers (H0: Stationary) ===')
print(f'Test Statistic: {kpss_raw.statistic:.4f}')
print(f'P-value:        {kpss_raw.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Non-stationary)" if kpss_raw.reject_at_5pct else "Fail to reject H0 (Stationary)"}')

kpss_log = kpss_test(y_log, regression='c')
print(f'\n=== KPSS Test: Log(Passengers) (H0: Stationary) ===')
print(f'Test Statistic: {kpss_log.statistic:.4f}')
print(f'P-value:        {kpss_log.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Non-stationary)" if kpss_log.reject_at_5pct else "Fail to reject H0 (Stationary)"}')

In [7]:
# Zivot-Andrews Test (structural break)
za_raw = zivot_andrews_test(y, model='c', trim=0.15)
print('=== Zivot-Andrews Test: Raw Passengers ===')
print(f'Test Statistic: {za_raw.statistic:.4f}')
print(f'P-value:        {za_raw.pvalue}')
print(f'Decision:       {"Reject H0 (Stationary with break)" if za_raw.reject_at_5pct else "Fail to reject H0 (Unit root)"}')
print(f'Critical Values: {za_raw.critical_values}')

In [8]:
# Test on differenced series (d=1 and D=1)
dy_log = np.diff(y_log)
dy_log_12 = np.diff(y_log, 12)
ddy_log = np.diff(np.diff(y_log, 12))

print('=== Tests on Transformed Series ===')

adf_dlog = adf_test(dy_log, regression='c', autolag='AIC')
print(f'\nADF on diff(log(y)): stat={adf_dlog.statistic:.4f}, p={adf_dlog.pvalue:.4f} '
      f'=> {"Stationary" if adf_dlog.reject_at_5pct else "Non-stationary"}')

adf_d12 = adf_test(dy_log_12, regression='c', autolag='AIC')
print(f'ADF on diff12(log(y)): stat={adf_d12.statistic:.4f}, p={adf_d12.pvalue:.4f} '
      f'=> {"Stationary" if adf_d12.reject_at_5pct else "Non-stationary"}')

adf_dd = adf_test(ddy_log, regression='c', autolag='AIC')
print(f'ADF on diff(diff12(log(y))): stat={adf_dd.statistic:.4f}, p={adf_dd.pvalue:.4f} '
      f'=> {"Stationary" if adf_dd.reject_at_5pct else "Non-stationary"}')

print('\n=> Conclusion: Series is I(1) with seasonal unit root.')
print('   Use d=1, D=1 for seasonal ARIMA models.')
print('   Log transformation recommended for multiplicative seasonality.')

In [9]:
# Save test results to JSON
tests_results = {
    'dataset': 'airline.csv',
    'n_obs': len(y),
    'frequency': 'monthly',
    'tests': {
        'adf_raw': {
            'statistic': float(adf_raw.statistic),
            'pvalue': float(adf_raw.pvalue),
            'reject_h0': bool(adf_raw.reject_at_5pct),
            'conclusion': 'Non-stationary' if not adf_raw.reject_at_5pct else 'Stationary'
        },
        'adf_log': {
            'statistic': float(adf_log.statistic),
            'pvalue': float(adf_log.pvalue),
            'reject_h0': bool(adf_log.reject_at_5pct),
            'conclusion': 'Non-stationary' if not adf_log.reject_at_5pct else 'Stationary'
        },
        'kpss_raw': {
            'statistic': float(kpss_raw.statistic),
            'pvalue': float(kpss_raw.pvalue),
            'reject_h0': bool(kpss_raw.reject_at_5pct),
            'conclusion': 'Non-stationary' if kpss_raw.reject_at_5pct else 'Stationary'
        },
        'kpss_log': {
            'statistic': float(kpss_log.statistic),
            'pvalue': float(kpss_log.pvalue),
            'reject_h0': bool(kpss_log.reject_at_5pct),
            'conclusion': 'Non-stationary' if kpss_log.reject_at_5pct else 'Stationary'
        },
        'zivot_andrews_raw': {
            'statistic': float(za_raw.statistic),
            'pvalue': float(za_raw.pvalue) if za_raw.pvalue is not None else None,
            'reject_h0': bool(za_raw.reject_at_5pct)
        },
        'adf_diff_log': {
            'statistic': float(adf_dlog.statistic),
            'pvalue': float(adf_dlog.pvalue),
            'reject_h0': bool(adf_dlog.reject_at_5pct),
            'conclusion': 'Stationary' if adf_dlog.reject_at_5pct else 'Non-stationary'
        },
        'adf_diff_seasonal_log': {
            'statistic': float(adf_dd.statistic),
            'pvalue': float(adf_dd.pvalue),
            'reject_h0': bool(adf_dd.reject_at_5pct),
            'conclusion': 'Stationary' if adf_dd.reject_at_5pct else 'Non-stationary'
        }
    },
    'integration_order': {
        'regular': 1,
        'seasonal': 1,
        'log_transform': True
    }
}

with open(os.path.join(output_dir, 'univariate_tests.json'), 'w') as f:
    json.dump(tests_results, f, indent=2)

print(f'Tests saved to {output_dir}/univariate_tests.json')

---
## 3. STL Decomposition

We apply STL decomposition to the log-transformed series with `period=12` to capture
the monthly seasonal pattern. Using the log scale linearizes the multiplicative
seasonality into an additive pattern.

In [10]:
# STL decomposition on log-transformed series
stl = STL(period=12, seasonal=7, robust=True)
stl_result = stl.fit(y_log)

print('=== STL Decomposition (log scale) ===')
print(f'Period:         {stl_result.period}')
print(f'Trend range:    [{stl_result.trend.min():.4f}, {stl_result.trend.max():.4f}]')
print(f'Seasonal range: [{stl_result.seasonal.min():.4f}, {stl_result.seasonal.max():.4f}]')
print(f'Remainder std:  {stl_result.remainder.std():.4f}')

# Seasonal strength
var_remainder = np.var(stl_result.remainder)
var_detrended = np.var(stl_result.seasonal + stl_result.remainder)
seasonal_strength = 1 - var_remainder / var_detrended
print(f'\nSeasonal strength: {seasonal_strength:.4f}')
print(f'  => {"Strong" if seasonal_strength > 0.6 else "Weak"} seasonality')

In [11]:
# GRAPH 2: STL decomposition
fig = plot_decomposition(
    results=stl_result,
    title='STL Decomposition - Log(Airline Passengers)',
    figsize=(12, 10)
)
plt.tight_layout()
plt.show()

---
## 4. Economic Filters

We apply HP and Hamilton filters to extract trend and cyclical components.
For monthly data, the HP filter uses lambda=14400 (Ravn-Uhlig rule).

In [12]:
# HP Filter on log passengers
hp_trend, hp_cycle = hp_filter(y_log, frequency='monthly')

print('=== HP Filter (lambda=14400 for monthly) ===')
print(f'Cycle std:       {hp_cycle.std():.4f}')
print(f'Cycle mean:      {hp_cycle.mean():.4f}')
print(f'Max expansion:   {hp_cycle.max():.4f}')
print(f'Max contraction: {hp_cycle.min():.4f}')

In [13]:
# GRAPH 3: HP Filter
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df.index, y_log, label='Log(Passengers)', color='steelblue', alpha=0.7)
axes[0].plot(df.index, hp_trend, label='HP Trend', color='red', linewidth=2)
axes[0].set_title('HP Filter: Trend Extraction (Log Scale)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(df.index, hp_cycle, 0,
                     where=hp_cycle >= 0, color='green', alpha=0.3, label='Above trend')
axes[1].fill_between(df.index, hp_cycle, 0,
                     where=hp_cycle < 0, color='red', alpha=0.3, label='Below trend')
axes[1].plot(df.index, hp_cycle, color='black', linewidth=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('HP Filter: Cyclical Component', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [14]:
# Hamilton Filter
ham_trend, ham_cycle = hamilton_filter(y_log, h=24, p=12)

print('=== Hamilton Filter (h=24, p=12 for monthly) ===')
valid = ~np.isnan(ham_cycle)
print(f'Valid observations: {valid.sum()}')
print(f'Cycle std:  {np.nanstd(ham_cycle):.4f}')

# Comparison plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, hp_cycle, label='HP Cycle', color='steelblue', linewidth=1.5)
ax.plot(df.index[:len(ham_cycle)], ham_cycle, label='Hamilton Cycle',
        color='darkorange', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('HP vs Hamilton Filter: Cycle Comparison', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Model Fitting

We fit 6 models to the airline data and compare their performance:

1. **SARIMA(0,1,1)(0,1,1)[12]** - Classic Box-Jenkins airline model
2. **SARIMA(1,1,0)(1,1,0)[12]** - Seasonal AR alternative
3. **ETS(M,A,M)** - Multiplicative error and seasonality, additive trend
4. **Auto-ARIMA** - Automatic selection with `seasonal=True, m=12`
5. **Holt-Winters** - Multiplicative seasonal smoothing
6. **Theta** - Theta method (non-seasonal baseline)

We use the last 12 months as the test set.

In [15]:
# Train/test split
h = 12
y_train = y[:-h]
y_test = y[-h:]

print(f'Training set: {len(y_train)} observations ({df.index[0].date()} to {df.index[-h-1].date()})')
print(f'Test set:     {len(y_test)} observations ({df.index[-h].date()} to {df.index[-1].date()})')

In [16]:
# Model 1: SARIMA(0,1,1)(0,1,1)[12] - Box-Jenkins classic
model_bj = ARIMA(order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
result_bj = model_bj.fit(y_train)

print('=== SARIMA(0,1,1)(0,1,1)[12] - Box-Jenkins Classic ===')
print(result_bj.summary())

In [17]:
# Model 2: SARIMA(1,1,0)(1,1,0)[12]
model_sar = ARIMA(order=(1, 1, 0), seasonal_order=(1, 1, 0, 12))
result_sar = model_sar.fit(y_train)

print('=== SARIMA(1,1,0)(1,1,0)[12] ===')
print(result_sar.summary())

In [18]:
# Model 3: ETS(M,A,M) - Multiplicative error & seasonal
model_ets = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
result_ets = model_ets.fit(y_train)

print('=== ETS(M,A,M) ===')
print(result_ets.summary())

In [19]:
# Model 4: Auto-ARIMA with seasonal detection
result_auto = auto_arima(
    y_train,
    seasonal=True,
    m=12,
    max_p=3, max_q=3, max_d=2,
    max_P=2, max_Q=2, max_D=1,
    information_criterion='aicc',
    stepwise=True,
    trace=True
)

print(f'\n=== Auto-ARIMA Selected: {result_auto.model_name} ===')
print(result_auto.summary())

In [20]:
# Model 5: Holt-Winters multiplicative
model_hw = HoltWinters(seasonal='multiplicative', seasonal_period=12)
result_hw = model_hw.fit(y_train)

print('=== Holt-Winters (Multiplicative) ===')
print(result_hw.summary())

In [21]:
# Model 6: Theta method (non-seasonal baseline)
model_theta = ThetaMethod()
result_theta = model_theta.fit(y_train)

print('=== Theta Method ===')
print(result_theta.summary())

---
## 6. Model Comparison

We compare models using:
- **In-sample**: AIC, BIC
- **Out-of-sample**: RMSE, MAE, MAPE on the 12-month test set
- **Final ranking** by RMSE (primary metric)

In [22]:
# Generate forecasts and compute metrics
models = {
    'SARIMA(0,1,1)(0,1,1)[12]': result_bj,
    'SARIMA(1,1,0)(1,1,0)[12]': result_sar,
    'ETS(M,A,M)': result_ets,
    f'Auto-ARIMA ({result_auto.model_name})': result_auto,
    'Holt-Winters (mult)': result_hw,
    'Theta': result_theta,
}

comparison = []
forecasts = {}

for name, result in models.items():
    # Forecast h steps ahead
    fc = result.forecast(steps=h)

    # ARIMA/auto_arima returns dict {'forecast', 'lower', 'upper'}
    # ETS/HW/Theta returns ndarray directly
    if isinstance(fc, dict):
        fc_mean = fc['forecast']
    else:
        fc_mean = np.asarray(fc).flatten()[:h]

    forecasts[name] = fc_mean

    # Out-of-sample metrics
    errors = y_test - fc_mean
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    mape = np.mean(np.abs(errors / y_test)) * 100

    # In-sample metrics
    aic_val = getattr(result, 'aic', None)
    bic_val = getattr(result, 'bic', None)

    comparison.append({
        'Model': name,
        'AIC': float(aic_val) if aic_val is not None else np.nan,
        'BIC': float(bic_val) if bic_val is not None else np.nan,
        'RMSE': float(rmse),
        'MAE': float(mae),
        'MAPE (%)': float(mape),
    })

comp_df = pd.DataFrame(comparison).set_index('Model')
print('=== Model Comparison ===')
print(comp_df.round(4).to_string())

In [23]:
# Save model comparison to CSV
comp_df.to_csv(os.path.join(output_dir, 'univariate_models_comparison.csv'))
print(f'Saved: {output_dir}/univariate_models_comparison.csv')

In [24]:
# GRAPH 4: Out-of-sample forecast comparison
fig, ax = plt.subplots(figsize=(14, 7))

n_show = 24
idx_train = df.index[-(h + n_show):-h]
idx_test = df.index[-h:]

ax.plot(idx_train, y_train[-n_show:], color='black', linewidth=1.5, label='Training')
ax.plot(idx_test, y_test, color='black', linewidth=2.5, linestyle='--', label='Actual')

colors = ['steelblue', 'darkorange', 'green', 'red', 'purple', 'brown']
for (name, fc_mean), color in zip(forecasts.items(), colors):
    ax.plot(idx_test, fc_mean, color=color, linewidth=1.2, label=name,
            marker='o', markersize=3)

ax.axvline(x=idx_test[0], color='gray', linestyle=':', alpha=0.5)
ax.set_title('Out-of-Sample Forecast Comparison (12-month holdout)', fontsize=14)
ax.set_ylabel('Passengers (thousands)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [25]:
# GRAPH 5: Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ['AIC', 'RMSE', 'MAPE (%)']
for ax, metric in zip(axes, metrics_to_plot):
    values = comp_df[metric].dropna()
    colors_bar = ['green' if v == values.min() else 'steelblue' for v in values]
    ax.barh(range(len(values)), values, color=colors_bar, alpha=0.8)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(values.index, fontsize=8)
    ax.set_title(metric, fontsize=12)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Model Comparison (green = best)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [26]:
# === RANKING TABLE: Models sorted by RMSE ===
ranking = comp_df.sort_values('RMSE').copy()
ranking.insert(0, 'Rank', range(1, len(ranking) + 1))

print('=' * 80)
print('     FINAL MODEL RANKING (sorted by RMSE)')
print('=' * 80)
print(ranking.round(4).to_string())
print('=' * 80)

best_name = ranking.index[0]
print(f'\nBest model: {best_name}')
print(f'  RMSE:    {ranking.loc[best_name, "RMSE"]:.2f}')
print(f'  MAE:     {ranking.loc[best_name, "MAE"]:.2f}')
print(f'  MAPE:    {ranking.loc[best_name, "MAPE (%)"]:.2f}%')

# Is seasonal component important?
seasonal_models = [n for n in ranking.index if 'Theta' not in n]
non_seasonal = [n for n in ranking.index if 'Theta' in n]
if non_seasonal:
    theta_rmse = ranking.loc[non_seasonal[0], 'RMSE']
    best_seasonal_rmse = ranking.loc[seasonal_models[0], 'RMSE']
    improvement = (theta_rmse - best_seasonal_rmse) / theta_rmse * 100
    print(f'\n=> Seasonal models improve over Theta by {improvement:.1f}% in RMSE')
    print(f'   => The seasonal component is {"very important" if improvement > 20 else "important"}!')

---
## 7. Diagnostics of the Best Model

In [27]:
# Select best model
best_result = models[best_name]

print(f'Best model: {best_name}')

# Residual diagnostics
resid = getattr(best_result, 'residuals', None) or getattr(best_result, 'resid', None)

lb = ljung_box_test(resid, lags=12)
jb = jarque_bera_test(resid)

print(f'\n=== Residual Diagnostics ===')
print(f'Residuals: n={len(resid)}, mean={np.mean(resid):.4f}, std={np.std(resid):.4f}')
print(f'\nLjung-Box (lag=12): stat={lb.statistic:.4f}, p={lb.pvalue:.4f}')
print(f'  -> {"No autocorrelation (good)" if not lb.reject_at_5pct else "Autocorrelation detected"}')
print(f'Jarque-Bera:        stat={jb.statistic:.4f}, p={jb.pvalue:.4f}')
print(f'  -> {"Normal residuals (good)" if not jb.reject_at_5pct else "Non-normal residuals"}')

In [28]:
# GRAPH 6: Residual diagnostics
fig = plot_diagnostics(
    results=best_result,
    lags=24,
    title=f'Residual Diagnostics - {best_name}',
    figsize=(12, 8)
)
plt.tight_layout()
plt.show()

---
## 8. Forecasting with the Best Model

Refit the best model on full data and forecast 24 months ahead with 95% CI.

In [29]:
# Refit best model on full data
if 'SARIMA(0,1,1)(0,1,1)' in best_name:
    best_model_full = ARIMA(order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
elif 'SARIMA(1,1,0)(1,1,0)' in best_name:
    best_model_full = ARIMA(order=(1, 1, 0), seasonal_order=(1, 1, 0, 12))
elif 'Auto-ARIMA' in best_name:
    best_model_full = ARIMA(
        order=best_result.model.order,
        seasonal_order=best_result.model.seasonal_order
    )
elif 'ETS' in best_name:
    best_model_full = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
elif 'Holt-Winters' in best_name:
    best_model_full = HoltWinters(seasonal='multiplicative', seasonal_period=12)
else:
    best_model_full = ThetaMethod()

best_result_full = best_model_full.fit(y)

h_fwd = 24
# ARIMA supports alpha for CI; ETS/HW/Theta do not
try:
    fc_full = best_result_full.forecast(steps=h_fwd, alpha=0.05)
except TypeError:
    fc_full = best_result_full.forecast(steps=h_fwd)

if isinstance(fc_full, dict):
    fc_mean = fc_full['forecast']
    fc_lower = fc_full.get('lower', fc_mean * 0.85)
    fc_upper = fc_full.get('upper', fc_mean * 1.15)
else:
    fc_mean = np.asarray(fc_full).flatten()[:h_fwd]
    # Approximate CI from residual std
    resid = getattr(best_result_full, 'resid', getattr(best_result_full, 'residuals', None))
    if resid is not None:
        sigma = np.std(resid)
        fc_lower = fc_mean - 1.96 * sigma * np.sqrt(np.arange(1, h_fwd + 1))
        fc_upper = fc_mean + 1.96 * sigma * np.sqrt(np.arange(1, h_fwd + 1))
    else:
        fc_lower = fc_mean * 0.85
        fc_upper = fc_mean * 1.15

fc_dates = pd.date_range(
    start=df.index[-1] + pd.DateOffset(months=1),
    periods=h_fwd, freq='MS'
)

print(f'=== {h_fwd}-Month Ahead Forecast ({best_name}) ===')
fc_table = pd.DataFrame({
    'Date': fc_dates,
    'Forecast': fc_mean,
    'Lower_95': fc_lower,
    'Upper_95': fc_upper
}).set_index('Date')
print(fc_table.round(2).to_string())

In [30]:
# GRAPH 7: Final forecast with confidence intervals
fig, ax = plt.subplots(figsize=(14, 7))

n_hist = 48
hist_idx = df.index[-n_hist:]
ax.plot(hist_idx, y[-n_hist:], color='steelblue', linewidth=1.5, label='Historical')

ax.plot(fc_dates, fc_mean, color='red', linewidth=2, label='Forecast')
ax.fill_between(fc_dates, fc_lower, fc_upper, color='red', alpha=0.15, label='95% CI')

ax.axvline(x=df.index[-1], color='gray', linestyle=':', alpha=0.5)
ax.set_title(f'Airline Passengers Forecast - {best_name}', fontsize=14)
ax.set_ylabel('Passengers (thousands)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [31]:
# Save forecasts to CSV
fc_save = pd.DataFrame({
    'date': fc_dates.strftime('%Y-%m-%d'),
    'forecast': fc_mean,
    'lower_95': fc_lower,
    'upper_95': fc_upper,
    'model': best_name
})

# Also save out-of-sample forecasts
oos_records = []
for name, fc_vals in forecasts.items():
    for i in range(len(fc_vals)):
        oos_records.append({
            'date': df.index[-h + i].strftime('%Y-%m-%d'),
            'model': name,
            'forecast': float(fc_vals[i]),
            'actual': float(y_test[i]),
            'error': float(y_test[i] - fc_vals[i])
        })

fc_save.to_csv(os.path.join(output_dir, 'univariate_forecasts.csv'), index=False)
print(f'Saved: {output_dir}/univariate_forecasts.csv')

---
## 9. Report Generation

In [32]:
# Generate comparison report
python_results = {
    'best_model': best_name,
    'rmse': float(comp_df.loc[best_name, 'RMSE']),
    'mae': float(comp_df.loc[best_name, 'MAE']),
    'mape': float(comp_df.loc[best_name, 'MAPE (%)']),
    'n_models_compared': len(models),
    'test_set_size': h,
    'forecast_horizon': h_fwd,
}

ranking_html = '<h3>Model Ranking (by RMSE)</h3>'
ranking_html += '<table><tr><th>Rank</th><th>Model</th><th>RMSE</th><th>MAE</th><th>MAPE</th></tr>'
for i, (name, row) in enumerate(comp_df.sort_values('RMSE').iterrows()):
    ranking_html += (f'<tr><td>{i+1}</td><td>{name}</td>'
                     f'<td>{row["RMSE"]:.2f}</td><td>{row["MAE"]:.2f}</td>'
                     f'<td>{row["MAPE (%)"]:.2f}%</td></tr>')
ranking_html += '</table>'

report_path = generate_comparison_report(
    title='Univariate Workflow Solution: Airline Passengers',
    python_results=python_results,
    output_path=os.path.join(output_dir, 'univariate_report.html'),
    extra_sections=[{'title': 'Model Ranking', 'html': ranking_html}]
)

print(f'Report generated: {report_path}')

---
## 10. Conclusions and Recommendations

### Key Findings

1. **Stationarity**: The airline passengers series is non-stationary in both level and
   variance. ADF and KPSS tests confirm I(1) behavior. Log transformation stabilizes
   the variance, and seasonal differencing is needed (D=1).

2. **Seasonality**: STL decomposition reveals very strong monthly seasonality
   (seasonal strength > 0.9). Summer months (July-August) consistently show the
   highest passenger counts. The seasonal amplitude grows with the level,
   confirming multiplicative seasonality.

3. **Best Model**: Among 6 candidate models, the seasonal models significantly
   outperform the non-seasonal Theta method. The classic Box-Jenkins
   SARIMA(0,1,1)(0,1,1)[12] or the Holt-Winters multiplicative model typically
   perform best on this dataset.

4. **Seasonal Component**: The seasonal component is **critical** for this dataset.
   Models that explicitly capture monthly seasonality reduce forecast error by
   20-50% compared to the non-seasonal baseline.

### Recommendations

- **Always test for seasonality** before choosing a model.
- **Log transformation** should be applied when seasonality is multiplicative.
- **Multiple model comparison** is essential. No single model dominates universally.
- **Out-of-sample evaluation** is more informative than in-sample criteria alone.

---
**End of Solution Notebook**

All intermediate results saved to `outputs/`:
- `univariate_tests.json`
- `univariate_models_comparison.csv`
- `univariate_forecasts.csv`